# Practice 29 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: `observed=True` in `pivot_table()`

You've already seen `observed=True` in `groupby()`. The same issue applies to `pivot_table()` — when the index or columns are categorical, pandas by default includes **all possible categories** as rows/columns, even ones with no data.

```python
# Without observed=True — shows empty rows for unused categories, FutureWarning
df.pivot_table(index='cut', columns='color', values='price')

# With observed=True — only shows categories that actually appear in the data
df.pivot_table(index='cut', columns='color', values='price', observed=True)
```

Always add `observed=True` when your index or columns are `category` dtype.

---
### Task 1: tips — pivot with category columns

Load `sns.load_dataset('tips')`. Convert `day`, `time`, and `smoker` to `category`.

1. Build a pivot table: mean `tip` by `day` × `smoker`. Run it **without** `observed=True` first — what do you notice? Then add `observed=True`.
2. Build a second pivot table: mean `total_bill` by `time` × `day`, with `observed=True`. Stack it into a Series with a MultiIndex. Use `.xs()` to extract all Dinner rows.
3. Use `.cat.remove_unused_categories()` on `day` after filtering to just weekend rows (`'Sat'` and `'Sun'`). Confirm the weekday categories are gone.
4. List comprehension: collect columns that are `category` dtype.

In [24]:
# Your code here
tips = sns.load_dataset('tips')
tips[['day','time','smoker']] = tips[['day','time','smoker']].astype('category')
p = tips.pivot_table(
    index= 'day',
    columns='smoker',
    values='tip'
)
p

p1 = tips.pivot_table(
    index= 'day',
    columns='smoker',
    values='tip',
    observed=True
)
p1

p2 = tips.pivot_table(
    index = 'time',
    columns='day',
    values='total_bill',
    observed=True
).stack()
dinner = p2.xs('Dinner',level='time')
dinner

weekend = tips[tips['day'].isin(['Sat','Sun'])].copy()
weekend['day'] = weekend['day'].cat.remove_unused_categories()
weekend['day']


cols = [col for col in tips.columns if tips[col].dtype == 'category']
cols


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_86645/2210011262.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  p = tips.pivot_table(


['sex', 'smoker', 'day', 'time']

---
## Level 2 — Transformations

### Task 2: mpg — answer these questions

Load `sns.load_dataset('mpg')`. The dataset has car fuel efficiency data: columns include `mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year`, `origin`, and `name`.

`cylinders` takes values 3, 4, 5, 6, 8 — treat it as an ordered category (3 < 4 < 5 < 6 < 8). `origin` (usa / europe / japan) is an unordered category.

Answer these questions — no sub-steps, use whatever approach you prefer:

1. Convert `cylinders` and `origin` to the appropriate category dtypes. How much memory did you save vs the original?
2. Use `.loc` with a boolean mask to select only high-cylinder cars (strictly more than 4 cylinders). What fraction of those are from the USA?
3. Add a `power_to_weight` column (`horsepower / weight`). Which origin has the highest mean power_to_weight? Use NumPy.
4. Among 4-cylinder cars only, which `model_year` had the best average mpg? Drop NaNs before computing.

In [40]:
# Your code here

mpg = sns.load_dataset('mpg')
co = pd.CategoricalDtype(categories=[3,4,5,6,8],
                         ordered=True)
mpg1 = mpg.copy()
mpg1['cylinders'] = mpg1['cylinders'].astype(co)
mpg1['origin'] = mpg1['origin'].astype('category')

mpg.memory_usage(deep=True).sum() - mpg1.memory_usage(deep=True).sum()

(mpg1.loc[mpg1['cylinders']>4]['origin']=='usa').mean()

mpg1['power_to_weight'] = mpg1['horsepower']/mpg1['weight']
m = mpg1.groupby('origin', observed=True)['power_to_weight'].mean()
m.index[np.argmax(m)]

c4 = mpg1[mpg1['cylinders']==4].dropna()
c4.groupby('model_year')['mpg'].mean().idxmax()


np.int64(80)

---
## Level 3 — Aggregation

### Task 3: penguins — pipe and summarise

Load `sns.load_dataset('penguins')`. Drop rows with any nulls.

Write a `.pipe()` chain with 3 functions (no starters). Requirements:
- One converts at least two columns to `category`
- One adds a `bill_ratio` column (`bill_length_mm / bill_depth_mm`)
- One adds a species-level average `bill_ratio` using `transform`

After the chain:
- Which species has the highest mean `bill_ratio`? Don't use `.idxmax()` — use `np.argmax()` instead.
- Build a pivot table: mean `bill_ratio` by `species` × `island`, with `observed=True`. Which combinations have no data (NaN)?
- Stack the pivot table and use `.xs()` to pull out all Chinstrap rows.

In [64]:
# Your code here
penguins = sns.load_dataset('penguins').dropna()

def conv_cat(df):
    df['island'] = df['island'].astype('category')
    df['species'] = df['species'].astype('category')
    return df

def add_ratio(df):
    df['bill_ratio'] = df['bill_length_mm']/df['bill_depth_mm']
    return df

def add_ra(df):
    df['species_av_br'] = df.groupby('species',observed = True)['bill_ratio'].transform('mean')
    return df 

res = (
    penguins.copy()
    .pipe(conv_cat)
    .pipe(add_ratio)
    .pipe(add_ra)
)

m = res.groupby('species', observed=True)['bill_ratio'].mean()
m.index[np.argmax(m)]

p = res.pivot_table(
    index = 'species',
    columns = 'island',
    values = 'bill_ratio',
    observed = True
)

[(r,c) for r in p.index for c in p.columns if pd.isna(p.loc[r,c])]


ps = p.stack()
ps.xs('Chinstrap', level='species')


island
Dream    2.653756
dtype: float64

---
## Level 4 — Real-world

### Task 4: titanic — open analysis

Load `sns.load_dataset('titanic')`. No scaffolding.

Build a complete analysis with these requirements:
- At least two columns converted to `category` (one must be ordered)
- Use `.loc` for at least one filter
- A `.pipe()` chain that adds two new columns
- A reshaped summary (pivot table or stack/unstack) with `observed=True`
- Extract a slice with `.xs()`
- Answer one question using NumPy — your choice

In [87]:
# Your code here
titanic = sns.load_dataset('titanic')
po = pd.CategoricalDtype(categories=[3,2,1],
                         ordered=True)
co = pd.CategoricalDtype(categories=['Third','Second','First'],
                         ordered=True)
titanic['pclass'] = titanic['pclass'].astype(po)
titanic['class'] = titanic['class'].astype(co)


titanic.loc[titanic['class']>'Third']


def is_senior(df):
    df['senior'] = df['age']>60
    return df
def is_highFare(df):
    df['high_fare'] = df['fare']>50
    return df

titanic = (
    titanic
    .pipe(is_senior)
    .pipe(is_highFare)
)


#titp = titanic.set_index(['sex','class'])
tp = titanic.pivot_table(
    index = 'sex',
    columns = 'class',
    values = 'fare',
    observed = True
)
tp.stack().xs('female')

class
Third      16.118810
Second     21.970121
First     106.125798
dtype: float64

In [92]:
# question is male or female survived more?

m = titanic.groupby('sex')['survived'].mean()

m.index[np.argmax(m)]


'female'